# 13 Colab Joint Generation + Contrastive Training

Run-all Colab notebook for optional Stage 4 joint generation + contrastive training after the generation pipeline has produced checkpoints.



## 1. Runtime And Drive Mount

This notebook is designed for Colab Run all. It clones the repository into `/content/neurovlm`, copies your generated data from Google Drive into the clone, and writes run outputs back to Drive.


In [ ]:
import os, sys, json, time, platform, subprocess, shutil
from pathlib import Path

print('Python:', sys.version)
print('Platform:', platform.platform())
gpu = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(gpu.stdout if gpu.returncode == 0 else 'No GPU detected by nvidia-smi. Use Runtime -> Change runtime type -> GPU.')

from google.colab import drive
drive.mount('/content/drive')


## 2. Clone Or Pull Repository

In [ ]:
REPO_URL = 'https://github.com/neurovlm/neurovlm.git'
REPO_BRANCH = 'neurovlm_gnn'
REPO_DIR = Path('/content/neurovlm')

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', REPO_BRANCH], check=False)

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR / 'src'))
print('Repo ready at', REPO_DIR)
print(subprocess.run(['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip())


## 3. Install Dependencies

In [ ]:
%pip install -q -e /content/neurovlm
%pip install -q torch pandas numpy nibabel nilearn pyyaml tqdm safetensors transformers adapters pyarrow matplotlib scikit-learn


## 4. Link Generated Drive Data Into The Clone

This assumes your generated data lives under `/content/drive/MyDrive/neurovlm/` with `atlas_free_multipositive/cache/` and `data_ale_3dcnn/ale_caches/`. The notebook links those folders into `/content/neurovlm/` and rewrites JSONL `tensor_path` values to the actual ALE cache location.


In [ ]:
DRIVE_ROOT = Path('/content/drive/MyDrive')
DRIVE_PROJECT_DIR = DRIVE_ROOT / 'neurovlm'

if not DRIVE_PROJECT_DIR.exists():
    raise FileNotFoundError(f'Missing Drive project directory: {DRIVE_PROJECT_DIR}')

essential_drive_paths = [
    DRIVE_PROJECT_DIR / 'atlas_free_multipositive/cache/unified_jsonl/splits/train.jsonl',
    DRIVE_PROJECT_DIR / 'atlas_free_multipositive/cache/unified_jsonl/splits/val.jsonl',
    DRIVE_PROJECT_DIR / 'atlas_free_multipositive/cache/unified_jsonl/text_registry.jsonl',
    DRIVE_PROJECT_DIR / 'atlas_free_multipositive/cache/text_embeddings/specter_text_cache.pt',
    DRIVE_PROJECT_DIR / 'data_ale_3dcnn/ale_caches/atlas_free_4mm_fwhm9_crop_float16.pt',
]
missing_drive = [str(p) for p in essential_drive_paths if not p.exists()]
if missing_drive:
    raise FileNotFoundError('Missing required Drive files: ' + repr(missing_drive))

print('Using generated data from:', DRIVE_PROJECT_DIR)

# Link generated data into the cloned runtime repo. This avoids copying the big ALE cache.
links = {
    REPO_DIR / 'atlas_free_multipositive/cache':
        DRIVE_PROJECT_DIR / 'atlas_free_multipositive/cache',
    REPO_DIR / 'data_ale_3dcnn/ale_caches':
        DRIVE_PROJECT_DIR / 'data_ale_3dcnn/ale_caches',
}

for dst, src in links.items():
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() or dst.is_symlink():
        print('exists:', dst)
    else:
        dst.symlink_to(src, target_is_directory=True)
        print('linked:', dst, '->', src)

# Rewrite JSONL paths to the actual ALE cache location you have on Drive.
# This changes tensor_path from atlas_free_multipositive/data/ale_caches/... to data_ale_3dcnn/ale_caches/...
from atlas_free_multipositive.data_building.rewrite_jsonl_paths import rewrite_jsonl_paths

old_cache_dir = 'atlas_free_multipositive/data/ale_caches'
new_cache_dir = 'data_ale_3dcnn/ale_caches'
for split_jsonl in [
    REPO_DIR / 'atlas_free_multipositive/cache/unified_jsonl/splits/train.jsonl',
    REPO_DIR / 'atlas_free_multipositive/cache/unified_jsonl/splits/val.jsonl',
]:
    changed = rewrite_jsonl_paths(split_jsonl, old=old_cache_dir, new=new_cache_dir)
    print('rewrote', changed, 'tensor/nifti path values in', split_jsonl)

DRIVE_RUNS_DIR = DRIVE_PROJECT_DIR / 'atlas_free_multipositive/outputs/runs'
DRIVE_RUNS_DIR.mkdir(parents=True, exist_ok=True)
print('Drive run outputs:', DRIVE_RUNS_DIR)


## 5. Verify Local Training Files And JSONL References


In [ ]:
import json

required_local = [
    REPO_DIR / 'atlas_free_multipositive/cache/unified_jsonl/splits/train.jsonl',
    REPO_DIR / 'atlas_free_multipositive/cache/unified_jsonl/splits/val.jsonl',
    REPO_DIR / 'atlas_free_multipositive/cache/unified_jsonl/text_registry.jsonl',
    REPO_DIR / 'atlas_free_multipositive/cache/text_embeddings/specter_text_cache.pt',
    REPO_DIR / 'data_ale_3dcnn/ale_caches/atlas_free_4mm_fwhm9_crop_float16.pt',
]

missing = []
for p in required_local:
    ok = p.exists()
    print(f'{str(ok):5}  {p}')
    if not ok:
        missing.append(str(p))

# Validate actual tensor_path/nifti_path references used by the Dataset.
referenced = set()
for split in ['train', 'val']:
    split_path = REPO_DIR / f'atlas_free_multipositive/cache/unified_jsonl/splits/{split}.jsonl'
    with split_path.open() as f:
        for line in f:
            if not line.strip():
                continue
            row = json.loads(line)
            for key in ['tensor_path', 'nifti_path']:
                value = row.get(key)
                if value:
                    referenced.add(value)

for value in sorted(referenced):
    p = Path(value)
    p = p if p.is_absolute() else REPO_DIR / p
    if not p.exists():
        missing.append(str(p))

print('checked JSONL-referenced map/tensor paths:', len(referenced))
if missing:
    raise FileNotFoundError('Missing files after Drive link/path rewrite: ' + repr(missing[:20]))
print('All required files and JSONL-referenced paths are available.')


## 6. Locate Stage 1/2 Checkpoints

In [ ]:
import yaml
from atlas_free_multipositive.training.train_joint_generation_contrastive import train_from_config as train_joint
manifests = sorted(DRIVE_RUNS_DIR.glob('colab_generation_*/generation_stage_manifest.json'))
if not manifests:
    raise FileNotFoundError('Run 09_10_colab_generation_pipeline.ipynb first so Stage 1/2 checkpoints exist on Drive.')
manifest_path = manifests[-1]
manifest = json.loads(manifest_path.read_text())
AE_CHECKPOINT = manifest['autoencoder_checkpoint']
TEXT_PROJ_CHECKPOINT = manifest['random_text_to_brain_checkpoint']
print('manifest:', manifest_path)
print('AE_CHECKPOINT:', AE_CHECKPOINT)
print('TEXT_PROJ_CHECKPOINT:', TEXT_PROJ_CHECKPOINT)


## 7. Joint Generation + Contrastive Training

In [ ]:
joint_cfg = yaml.safe_load(Path('atlas_free_multipositive/configs/joint_generation_contrastive_config.yaml').read_text())
joint_run_dir = DRIVE_RUNS_DIR / time.strftime('colab_joint_%Y%m%d_%H%M%S')
joint_cfg.update({'autoencoder_checkpoint': AE_CHECKPOINT, 'text_projection_checkpoint': TEXT_PROJ_CHECKPOINT, 'checkpoint_dir': str(joint_run_dir / 'checkpoints'), 'device': 'auto', 'epochs': 3, 'batch_size': 4})
result = train_joint(joint_cfg)
joint_run_dir.mkdir(parents=True, exist_ok=True)
json.dump(result, open(joint_run_dir / 'joint_result.json', 'w'), indent=2)
print(json.dumps(result, indent=2))
